In [ ]:
import glob, os
import ast
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from astropy.io import fits
from scipy.optimize import curve_fit

%matplotlib widget

In [ ]:
def get_colors(n_steps, cmap_name='Blues', vmin=0, vmax=None):
    if vmax is None:
        vmax = n_steps - 1
    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.get_cmap(cmap_name)
    return [cmap(norm(i)) for i in range(n_steps)], norm, cmap

def has_key(hdul, key='THETA_R'):
    """Retourne True si THETA_R existe dans le header"""
    return key in hdul[0].header

def rename_header_key(fits_file, old_key, new_key, save=True):
    """Renomme une clé dans le header FITS"""
    with fits.open(fits_file, mode='update') as hdul:
        if old_key in hdul[0].header:
            value = hdul[0].header[old_key]
            hdul[0].header[new_key] = value
            del hdul[0].header[old_key]
            if save:
                hdul.flush()
            print(f"✓ Clé '{old_key}' renommée en '{new_key}' avec valeur: {value}")
            return True
        else:
            print(f"✗ Clé '{old_key}' non trouvée dans le header")
            return False
        
def get_chi2(data, model, error):
    chi2 = np.sum((data - model)**2 / error**2)
    return chi2


In [ ]:
#for file in glob.glob(path_save + "Scan_*.fits"):
 #   print(f"Fichier: {file}")
  #  rename_header_key(file, 'THETA_MA', 'ALPH_MA')
   # rename_header_key(file, 'THETA_MI', 'ALPH_MI')

# Play with cos functions

In [ ]:
xx = np.arange(0, 360)
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs = axs.ravel()
axs[0].plot(xx, np.cos(np.radians(xx)), label=('cos'))
axs[0].plot(xx, np.cos(np.radians(xx))**2, label=('cos^2'))
axs[0].plot(xx, np.cos(np.radians(xx))**4, label=('cos^4'))
axs[0].grid()
axs[0].legend()

axs[1].plot(xx, np.sin(np.radians(xx)), label=('sin'))
axs[1].plot(xx, np.sin(np.radians(xx))**2, label=('sin^2'))
axs[1].plot(xx, np.sin(np.radians(xx))**4, label=('sin^4'))
axs[1].grid()
axs[1].legend()

fig.tight_layout()

In [ ]:
trans = np.arange(0, 100, 80)
print(trans)
ff = np.size(trans)
colors, norm, cmap = get_colors(ff, cmap_name='viridis', vmin=0, vmax=ff-1)

xx = np.arange(0, 360)
plt.figure()
plt.plot(xx, np.cos(np.radians(xx))**4, label=('cos^4'), color='red')
plt.plot(xx, np.cos(np.radians(xx))**2 * np.sin(np.radians(xx))**2, label=('cos^2*sin^2'), color='orange')

for i in range(ff):
    #plt.plot(xx, np.cos(np.radians(trans[i]+xx))**2 * np.sin(np.radians(xx))**2, color=colors[i])
    plt.plot(xx, np.cos(np.radians(trans[i]+xx))**2 * np.cos(np.radians(xx))**2, color=colors[i])
    #plt.plot(xx, np.cos(np.radians(trans[i]-xx))**2 * np.sin(np.radians(xx-trans[i]))**2, color=colors[i])
plt.grid()
plt.legend()



# Tout premier test

On fait tourner le polariseur et on regarde la moyenne du signal.

In [ ]:

S21 = np.array([-75, -62, -52, -46, -42, -39, -37, -36, -35, -35, -35.5, -36.5, -38, -40.5, -44, -49, -56, -70, -75])
nmeas = np.size(S21)
angle = np.arange(nmeas)*10 +78



In [ ]:
tt = np.arange(70, 260)
plt.figure()
plt.plot(angle, S21, 'o')
#plt.plot(tt, (np.cos(np.radians(tt-180)))**4+np.mean(S21**2), 'r-')
plt.xlabel("Angle (°)")
plt.ylabel("S21 (dB)")
plt.grid(10)
plt.show()

# Scans automatisés

Les fichiers sont nommés Scan_Date_Time.fits

In [ ]:
path_save = '/home/lmousset/Projets_Recherche/COSMOCal/Tests_IAS_2026/Data/CosmoCal_data/'
#path_save = "C:\\Users\\Administrator\\Documents\\Scripts_Commande_VNA\\CosmoCal_data\\"

hdul = fits.open(path_save + "Scan_20260220_171128.fits")
header = hdul[0].header
header


In [ ]:
alpha_min = header['ALPH_MI']
alpha_max = header['ALPH_MA']
step = header['STEP']
nfreq = header['POINTS']

alpha = np.arange(alpha_min, alpha_max + step, step)
print(alpha.shape)

# For the first scans, THETA_R is not in the header but it was set to 0.0 during the acquisition, so we can use that as default value
if has_key(hdul, key='THETA_R'):
    thetaR = header['THETA_R']
else:
    thetaR = 0.0
print(thetaR)

In [ ]:
hdul.info()  # View structure
mag = hdul[0].data  # Access magnitude
phi = hdul[1].data  # Access phase

print(mag.shape)  # Check shape of magnitude data [(nstep, nS, points)]

mag_lin = 10**(mag/10)


## Plot the data

In [ ]:
#def cos4(alpha, thetaE=0, amp=1, offset=0):
 #   return amp *(np.cos(np.radians(thetaE - alpha)))**4 + offset

def cos4(alpha, thetaE=0, amp=1):
    return amp *(np.cos(np.radians(thetaE - alpha)))**4


In [ ]:
### Plot chaque fréquence

freqs = [0,2, 3, 10, 15, 20, 30, 40]
freqs = np.arange(60)
nf = len(freqs)
tt = np.arange(0, 380)
colors, norm, cmap = get_colors(nf, cmap_name='viridis', vmin=0, vmax=nf-1)
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax = ax.ravel()
for i, f in enumerate(freqs):
    ax[0].plot(alpha, mag_lin[:, 0, f], 'o', color=colors[i]) # S21
    ax[1].plot(alpha, mag_lin[:, 1, f], 'o', color=colors[i]) # S12

ax[0].set_xlabel("alpha (°)")
ax[0].set_ylabel("S21")
#ax[0].set_yscale('log')
ax[0].grid(10)

ax[1].set_xlabel("alpha (°)")
ax[1].set_ylabel("S12")
#ax[1].set_yscale('log')
ax[1].grid(10)
# créer une colorbar qui montre l'échelle d'indices (ou de valeurs réelles)
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # nécessaire pour colorbar
cbar = plt.colorbar(sm, ax=ax[0])
cbar = plt.colorbar(sm, ax=ax[1])
cbar.set_label('Indice de la fréquence')

fig.tight_layout()
fig.savefig(f"../Plots/Scan_allfreqs.png", dpi=300)


In [ ]:
# Plot la moyenne sur les fréquences
tt = np.arange(0, 380)
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax = ax.ravel()

ax[0].plot(alpha, np.mean(mag_lin[:, 0, :], axis=1), 'o', color='b')
ax[0].plot(tt, cos4(tt, 3)*np.max(np.mean(mag_lin[:, 0, :], axis=1)), 'orange')
ax[0].set_xlabel("alpha (°)")
ax[0].set_ylabel("S21")
#ax[0].set_yscale('log')
ax[0].grid(10)

ax[1].plot(alpha, np.mean(mag_lin[:, 1, :], axis=1), 'o', color= 'b')
ax[1].plot(tt, cos4(tt, 3)*np.max(np.mean(mag_lin[:, 1, :], axis=1)), 'orange')
ax[1].set_xlabel("alpha (°)")
ax[1].set_ylabel("S12")
#ax[1].set_yscale('log')
ax[1].grid(10)

fig.tight_layout()





## Estimation d'une erreur

On a fait N=5 scans identiques sans rien changer. Cela va nous permettre d'estimer une erreur.

Notre tableau de données a 4 dimensions : [i, t, a, f]
avec
* i l'indice du scan, on a fait N=5 scans
* t l'indice du coeff de scattering (0 pour S21 et 1 pour S12)
* a pour l'angle alpha du polariseur, on a 39 positions entre 0 et 380°
* f pour la fréquence, on a 60 fréquences

Nous allons calculer le STD sur les N scans.

In [ ]:
# Get the data
mag = []

os.chdir(path_save)

i = 0
for file in glob.glob("Scan_20260220_*.fits"):
    print(file)
    hdul = fits.open(path_save + file)
    header = hdul[0].header
    if has_key(hdul, key='THETA_R'):
        thetaR = header['THETA_R']
    else:
        thetaR = 0.0
    if thetaR == 0.0: # Get only the scan with THETA_R=0
        print(i, file)
        mag.append(hdul[0].data)  # Access magnitude
    i+=1

mag = np.array(mag)
print(mag.shape)

mag_lin = 10**(mag / 10)

nscans, nstep, nS, nfreq = mag_lin.shape

In [ ]:
mean_over_scans = np.mean(mag_lin, axis=0)
std_over_scans = np.std(mag_lin, axis=0)
print(std_over_scans.shape)

In [ ]:
f = 10

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax = ax.ravel()

### S21
ax[0].errorbar(alpha, mean_over_scans[:, 0, f], yerr=std_over_scans[:, 0, f], color='b', ls='None', marker='.')
ax[0].set_xlabel("alpha (°)")
ax[0].set_ylabel("Mean over scans")
ax[0].set_title(f"S21 - Frequency index: {f}")
#ax[0].set_yscale('log')
ax[0].grid(10)

### S12
ax[1].errorbar(alpha, mean_over_scans[:, 1, f], yerr=std_over_scans[:, 1, f], color='b', ls='None', marker='.')
ax[1].set_xlabel("alpha (°)")
ax[1].set_ylabel("Mean over scans")
ax[1].set_title(f"S12 - Frequency index: {f}")
#ax[1].set_yscale('log')
ax[1].grid(10)

fig.tight_layout()
fig.savefig(f"../Plots/Scan_MEAN_over_scans.png", dpi=300)


In [ ]:
ls

In [ ]:
f = 10

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax = ax.ravel()

### S21
ax[0].plot(alpha, std_over_scans[:, 0, f], '.', color='b')
ax[0].set_xlabel("alpha (°)")
ax[0].set_ylabel("STD over scans")
ax[0].set_title(f"S21 - Frequency index: {f}")
#ax[0].set_yscale('log')
ax[0].grid(10)

### S12
ax[1].plot(alpha, std_over_scans[:, 1, f], '.', color='b')
ax[1].set_xlabel("alpha (°)")
ax[1].set_ylabel("STD over scans")
ax[1].set_title(f"S12 - Frequency index: {f}")
#ax[1].set_yscale('log')
ax[1].grid(10)

fig.tight_layout()
fig.savefig(f"../Plots/Scan_STD_over_scans.png", dpi=300)

In [ ]:
nscans

In [ ]:
t = 0
plt.figure()
for f in range(10):
    plt.plot(mean_over_scans[:, t, f], std_over_scans[:, t, f]**2, 'o')

In [ ]:
mean_over_freq = np.mean(mag_lin, axis=3)
std_over_freq = np.std(mag_lin, axis=3)
print(std_over_freq.shape)


In [ ]:
i = 0 # Scan

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax = ax.ravel()

### S21
ax[0].errorbar(alpha, mean_over_freq[i, :, 0], yerr=std_over_freq[i, :, 0], color='r', ls='None', marker='.')
ax[0].set_xlabel("alpha (°)")
ax[0].set_ylabel("Mean over freq")
ax[0].set_title(f"S21 - Scan index: {i}")
#ax[0].set_yscale('log')
ax[0].grid(10)

### S12
ax[1].errorbar(alpha, mean_over_freq[i, :, 1], yerr=std_over_freq[i, :, 0], color='r', ls='None', marker='.')
ax[1].set_xlabel("alpha (°)")
ax[1].set_ylabel("Mean over freq")
ax[1].set_title(f"S12 - Scan index: {i}")
#ax[1].set_yscale('log')
ax[1].grid(10)

fig.tight_layout()
fig.savefig(f"../Plots/Scan_MEAN_over_freqs.png", dpi=300)

In [ ]:
i = 0 # Scan

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax = ax.ravel()

### S21
ax[0].plot(alpha, std_over_freq[i, :, 0], color='r', ls='None', marker='.')
ax[0].set_xlabel("alpha (°)")
ax[0].set_ylabel("STD over freq")
ax[0].set_title(f"S21 - Scan index: {i}")
#ax[0].set_yscale('log')
ax[0].grid(10)

### S12
ax[1].plot(alpha, std_over_freq[i, :, 0], color='r', ls='None', marker='.')
ax[1].set_xlabel("alpha (°)")
ax[1].set_ylabel("STD over freq")
ax[1].set_title(f"S12 - Scan index: {i}")
#ax[1].set_yscale('log')
ax[1].grid(10)

fig.tight_layout()
fig.savefig(f"../Plots/Scan_STD_over_freqs.png", dpi=300)

In [ ]:
std_over_freq.shape

In [ ]:
t = 0
fig = plt.figure()
for s in range(nscans):
    plt.plot(mean_over_freq[s, :, t], std_over_freq[s, :, t], 'o', label=f'Scan {s}')
plt.xlabel('Mean')
plt.ylabel('STD')
plt.legend()
fig.savefig(f"../Plots/Scan_STDvsMEAN.png", dpi=300)


## Ajustement de la moyenne sur les scans par fréquence

On considère que les scans sont indépendants. Donc on peut diviser le STD par sqrt(N) pour avoir l'erreur sur la moyenne.

In [ ]:
f = 12 # Frequency index 
t = 0 # trace index (S21 or S12)

signal = mean_over_scans[:, t, f]
sigma = std_over_scans[:, t, f] / np.sqrt(nscans)  # Erreur sur la moyenne

# Normalize to the max
#signal /= np.max(signal)

# Fit the data
popt, pcov = curve_fit(cos4, alpha, signal, sigma=sigma, absolute_sigma=True)
error = np.sqrt(np.diag(pcov))

print("Paramètres optimisés :", popt)
print("Erreurs sur les paramètres :", error)

# Residus et chi2
residuals = signal - cos4(alpha, *popt)
std_res = np.std(residuals)
print("Écart-type des résidus :", std_res)
chi2 = get_chi2(signal, cos4(alpha, *popt), sigma)
print("Chi2 :", chi2)

chi2_red = chi2 / (len(alpha) - len(popt))
print("Chi2 réduit :", chi2_red)

fig = plt.figure()
plt.errorbar(alpha, signal, yerr=sigma, marker='.', ls='None')
plt.plot(tt, cos4(tt, *popt), 'orange', label='Fit cos^4')
plt.xlabel("alpha (°)")
plt.ylabel("S21 mean over scans")
plt.title(r'$\theta_E$ = {:.3f} ± {:.3f}°'.format(popt[0], error[0]))
plt.grid(10)
fig.savefig(f"../Plots/Scan_Fit_freq{f}.png", dpi=300)

In [ ]:
# Loop over frequencies
t = 0 # trace index (S21 or S12)

allthetaE = []
allthetaE_err = []
allchi2_red = []

for f in range(60):
    signal = mean_over_scans[:, t, f]
    sigma = std_over_scans[:, t, f] / np.sqrt(nscans)  # Erreur sur la moyenne

    # Normalize to the max
    #signal /= np.max(signal)

    # Fit the data
    popt, pcov = curve_fit(cos4, alpha, signal, sigma=sigma, absolute_sigma=True)
    error = np.sqrt(np.diag(pcov))

    print("Paramètres optimisés :", popt)
    allthetaE.append(popt[0])
    print("Erreurs sur les paramètres :", error)
    allthetaE_err.append(error[0])

    # Residus et chi2
    residuals = signal - cos4(alpha, *popt)
    std_res = np.std(residuals)
    print("Écart-type des résidus :", std_res)
    chi2 = get_chi2(signal, cos4(alpha, *popt), sigma)
    print("Chi2 :", chi2)

    chi2_red = chi2 / (len(alpha) - len(popt))
    allchi2_red.append(chi2_red)
    print("Chi2 réduit :", chi2_red)

In [ ]:
np.array(np.round(allchi2_red))

In [ ]:
allthetaE = np.array(allthetaE)
allthetaE_err = np.array(allthetaE_err)

# Erreur sur la moyenne de thetaE
erreur = np.sqrt(np.sum(allthetaE_err**2 )) / nfreq
print(erreur)

mthetaE = np.mean(allthetaE)
print("ThetaE moyen :", mthetaE)
stdthetaE = np.std(allthetaE) / np.sqrt(nfreq)
print("Écart-type de thetaE :", stdthetaE)

fig = plt.figure()
plt.errorbar(np.arange(len(allthetaE)), allthetaE, yerr=allthetaE_err, fmt='.')
plt.axhline(mthetaE, color='orange', label=r'$Mean$')
plt.axhline(mthetaE-stdthetaE, color='orange', ls='--')
plt.axhline(mthetaE+stdthetaE, color='orange', label=r'$Mean \pm STD/\sqrt{N}$', ls='--')
plt.xlabel("Fréquence index")
plt.ylabel(r'$\theta_E$ (°)')
plt.grid(10)
plt.legend()
fig.savefig(f"../Plots/Scan_Fit_allthetaE", dpi=300)

# Ajustement de la moyenne

On ajuste la moyenne sur les fréquences et sur les scans.

In [ ]:
mean_over_freqscans = np.mean(mag_lin, axis=(0, 3))
std_over_freqscans = np.std(mag_lin, axis=(0, 3))
print(mean_over_freqscans.shape)

In [ ]:
# Take the mean
t = 0
signal = mean_over_freqscans[:, t]
sigma = std_over_freqscans[:, t] / (np.sqrt(nscans * nfreq))

# Normalize to the max
#signal /= np.max(signal)

# Fit the data
popt, pcov = curve_fit(cos4, alpha, signal, sigma=sigma, absolute_sigma=True)
error = np.sqrt(np.diag(pcov))

print("Paramètres optimisés :", popt)
print("Erreurs sur les paramètres :", error)

# Residus et chi2
residuals = signal - cos4(alpha, *popt)
std_res = np.std(residuals)
print("Écart-type des résidus :", std_res)

chi2 = get_chi2(signal, cos4(alpha, *popt), sigma)
print("Chi2 :", chi2)

chi2_red = chi2 / (len(alpha) - len(popt))
print("Chi2 réduit :", chi2_red)

fig = plt.figure()
plt.errorbar(alpha, signal, yerr=sigma, marker='.', ls='None')
plt.plot(tt, cos4(tt, *popt), 'orange', label='Fit cos^4')
plt.xlabel("alpha (°)")
plt.ylabel("S21 normalisé")
plt.title(r'$\theta_E$ = {:.3f} ± {:.3f}°'.format(popt[0], error[0]))
plt.grid(10)
plt.tight_layout()
fig.savefig(f"../Plots/Scan_Fit_mean.png", dpi=300)

In [ ]:
# Sans donner d'erreur au fit
# Take the mean
t = 0
signal = mean_over_freqscans[:, t]

# Fit the data
popt, pcov = curve_fit(cos4, alpha, signal, absolute_sigma=False)
error = np.sqrt(np.diag(pcov))

print("Paramètres optimisés :", popt)
print("Erreurs sur les paramètres :", error)

# Residus et chi2
residuals = signal - cos4(alpha, *popt)
std_res = np.std(residuals)
print("Écart-type des résidus :", std_res)
chi2 = get_chi2(signal, cos4(alpha, *popt), std_res)
print("Chi2 :", chi2)

chi2_red = chi2 / (len(alpha) - len(popt))
print("Chi2 réduit :", chi2_red)

plt.figure()
plt.errorbar(alpha, signal, marker='.', ls='None')
plt.plot(tt, cos4(tt, *popt), 'orange', label='Fit cos^4')
plt.xlabel("alpha (°)")
plt.ylabel("S21 normalisé")
plt.title(r'$\theta_E$ = {:.3f} ± {:.3f}°'.format(popt[0], error[0]))
plt.grid(10)

# Residus 

On soustrait le modèle calculé sur la moyenne à chaque fréquence pour essayer d'identifier une systématique.

In [ ]:
mag_lin.shape

In [ ]:
the_model = cos4(alpha, *popt)

In [ ]:
residus_freq = mag_lin - the_model[None, :, None, None]
print(residus_freq.shape)

In [ ]:
### Plot chaque fréquence
s = 0 # Scan

freqs = [0, 2, 3, 10, 15, 20, 30, 40]
nf = len(freqs)
tt = np.arange(0, 380)
colors, norm, cmap = get_colors(nf, cmap_name='viridis', vmin=0, vmax=nf-1)
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax = ax.ravel()
for i, f in enumerate(freqs):
    ax[0].plot(alpha, residus_freq[s, :, 0, f], 'o', color=colors[i]) # S21
    ax[1].plot(alpha, residus_freq[s, :, 1, f], 'o', color=colors[i]) # S12

ax[0].set_xlabel("alpha (°)")
ax[0].set_ylabel("S21 - fit")
#ax[0].set_yscale('log')
ax[0].grid(10)

ax[1].set_xlabel("alpha (°)")
ax[1].set_ylabel("S12 - fit")
#ax[1].set_yscale('log')
ax[1].grid(10)
# créer une colorbar qui montre l'échelle d'indices (ou de valeurs réelles)
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # nécessaire pour colorbar
cbar = plt.colorbar(sm, ax=ax[0])
cbar = plt.colorbar(sm, ax=ax[1])
cbar.set_label('Indice de la fréquence')

fig.tight_layout()